# Question Generation Training on RACE Dataset

**Goal:** Fine-tune Mistral 7B to generate multiple-choice questions from text passages (video transcripts)

**Dataset:** RACE (Reading Comprehension from Examinations)
- 100K multiple-choice questions from English exams
- Format: Passage → Question → 4 Choices → Correct Answer

**Output Format (for integration):**
```json
[
  {
    "question": "What is X?",
    "choices": ["A", "B", "C", "D"],
    "correct": 0
  }
]
```

## 1. Setup

In [ ]:
!pip install -q datasets transformers accelerate peft bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.3 MB/s eta 0:00:00


In [ ]:
!pip install trl
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import json

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 18.2 MB/s eta 0:00:00
CUDA: True
GPU: NVIDIA A100-SXM4-80GB


## 2. Load RACE Dataset

In [ ]:
# Load RACE dataset
print("Loading RACE dataset...")

# RACE has 'high' and 'middle' - load both
race_high = load_dataset("race", "high", split="train")
race_middle = load_dataset("race", "middle", split="train")

# Combine
from datasets import concatenate_datasets
race_train = concatenate_datasets([race_high, race_middle])

print(f"Total training examples: {len(race_train)}")
print(f"\nExample:")
print(race_train[0])

Loading RACE dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

high/test-00000-of-00001.parquet:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

high/train-00000-of-00001.parquet:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

...

middle/validation-00000-of-00001.parquet:   0%|          | 0.00/407k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1436 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/25421 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1436 [00:00<?, ? examples/s]

Total training examples: 87866

Example:
{'example_id': 'high19088.txt', 'article': 'Last week I talked with some of my students about what they wanted to do after they graduated, and what kind of job prospects  they thought they had.\nGiven that I teach students who are training to be doctors, I was surprised do find that most thought that they would not be able to get the jobs they wanted without "outside help". "What kind of help is that?" I asked, expecting them to tell me that they would need a   or family friend to help them out.\n"Surgery ," one replied.\nI was pretty alarmed by that response. It seems that the graduates of today are increasingly willing to go under the knife to get ahead of others when it comes to getting a job .\nOne girl told me that she was considering surgery to increase her height. "They break your legs, put in special extending screws, and slowly expand the gap between the two ends of the bone as it re-grows, you can get at least 5 cm taller!"\nAt that po

In [ ]:
# Explore RACE format
example = race_train[0]
print("RACE Format:")
print(f"Article: {example['article'][:200]}...")
print(f"\nQuestion: {example['question']}")
print(f"Options: {example['options']}")
print(f"Answer: {example['answer']}")

RACE Format:
Article: Last week I talked with some of my students about what they wanted to do after they graduated, and what kind of job prospects  they thought they had.
Given that I teach students who are training to be...

Question: We can know from the passage that the author works as a_.
Options: ['doctor', 'model', 'teacher', 'reporter']
Answer: C


## 3. Format Dataset for Training

In [ ]:
def format_question_generation(example):
    """
    Format RACE example for question generation training

    Input: Article (passage/transcript)
    Output: Generate question + choices + correct answer
    """
    article = example['article']
    question = example['question']
    options = example['options']
    answer = example['answer']  # Letter: A, B, C, or D

    # Convert answer letter to index
    answer_idx = ord(answer) - ord('A')

    # Format choices
    choices_text = "\n".join([f"{chr(65+i)}) {opt}" for i, opt in enumerate(options)])

    # Create prompt
    prompt = f"""### Passage:
{article}

### Task:
Generate a multiple-choice question based on the passage above.

### Question:
{question}

### Choices:
{choices_text}

### Correct Answer:
{answer}"""

    return {"text": prompt}

# Format dataset
print("Formatting dataset...")
formatted_dataset = race_train.map(format_question_generation, remove_columns=race_train.column_names)

# Sample and limit for faster training (use 20K examples)
formatted_dataset = formatted_dataset.shuffle(seed=42).select(range(20000))

print(f"Training on {len(formatted_dataset)} examples")
print(f"\nSample prompt:")
print(formatted_dataset[0]['text'][:500])

Formatting dataset...


Map:   0%|          | 0/87866 [00:00<?, ? examples/s]

Training on 20000 examples

Sample prompt:
### Passage:
THE human face doesn't lie. We show sadness and happiness through our expressions. But exactly how many emotions can our face make?Scientists used to believe we had six basic facial expressions that tell others how we feel: sad, happy, surprised, fearful, angry and disgusted  .
But a new study shows that our faces can do more than we think. Scientists from Ohio State University found out that humans can actually make 21 different facial expressions after studying how people move the


## 4. Load Base Model

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

base_model_name = "mistralai/Mistral-7B-v0.1"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)

print("Model loaded!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading tokenizer...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded!


## 5. Setup LoRA

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print("LoRA config created")

LoRA config created


## 6. Training Configuration

In [ ]:
# Training arguments (optimized for A100)
training_args = TrainingArguments(
    output_dir="./question-gen-checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=50,
    save_steps=500,
    save_total_limit=3,
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    weight_decay=0.001,
    report_to="none",
)

print("Training config set")

Training config set


## 7. Train Model

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=formatted_dataset,
    peft_config=lora_config,
    args=training_args,
)

Adding EOS to train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
print("Starting training...")
print(f"Total steps: ~{len(formatted_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")
print(f"Estimated time on A100: ~2-3 hours\n")

trainer.train()

Starting training...
Total steps: ~2500
Estimated time on A100: ~2-3 hours



Step,Training Loss
50,1.846949
100,1.742472
150,1.721646
200,1.718846
250,1.708511
300,1.717126
350,1.722628
400,1.697512
450,1.713172
500,1.693701


Step,Training Loss
50,1.846949
100,1.742472
150,1.721646
200,1.718846
250,1.708511
300,1.717126
350,1.722628
400,1.697512
450,1.713172
500,1.693701


TrainOutput(global_step=2500, training_loss=1.6206813934326172, metrics={'train_runtime': 8839.4756, 'train_samples_per_second': 4.525, 'train_steps_per_second': 0.283, 'total_flos': 1.1381811550291231e+18, 'train_loss': 1.6206813934326172})

## 8. Save Model

In [ ]:
# Save to Google Drive
output_dir = "/content/drive/MyDrive/question-generation-mistral"

print(f"Saving model to {output_dir}...")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("Model saved!")

Saving model to /content/drive/MyDrive/question-generation-mistral...
Model saved!


## 9. Test Question Generation

In [ ]:
# Test with a sample transcript
test_transcript = """
In this lecture, we'll discuss the concept of derivatives in calculus.
A derivative represents the rate of change of a function at a given point.
For example, if we have a function f(x) = x squared, the derivative is 2x.
This means that at any point x, the rate of change is twice the value of x.
Derivatives are fundamental in understanding how functions behave and are used
extensively in physics, engineering, and economics.
"""

prompt = f"""### Passage:
{test_transcript.strip()}

### Task:
Generate a multiple-choice question based on the passage above.

### Question:"""

# Generate
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("="*80)
print("GENERATED QUESTION:")
print("="*80)
print(generated_text)
print("="*80)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


GENERATED QUESTION:
### Passage:
In this lecture, we'll discuss the concept of derivatives in calculus.
A derivative represents the rate of change of a function at a given point.
For example, if we have a function f(x) = x squared, the derivative is 2x.
This means that at any point x, the rate of change is twice the value of x.
Derivatives are fundamental in understanding how functions behave and are used
extensively in physics, engineering, and economics.

### Task:
Generate a multiple-choice question based on the passage above.

### Question:
prefix = ###
###
prefix = ###  ###
 ### The most people who have been to ###
prefix = ###
###
 ###
###
###
###
###
BE and ### F. ###
 ###
 ###
prefix="t get the  ###  ###
A
Ý
###  ### C. ###
###
prefix = ###
Every package
 ###
Have I have many of ### The  ###
Do #20000, ###
 ###
 ### The G. ###
###
 ###
 ###
 prefixes of ###
###
 ###
###
 prefix = ###  ### The L. ###
BETR is a number of ###  ###     ### "s
prefix = ###
###
W_ ###
 ###
###
 ###
#

## 10. Integration Format Test

## 11. Generate Multiple Questions

In [19]:
def generate_questions(transcript, num_questions=5):
    """
    Generate multiple MCQ questions from a transcript
    This is the function you'll use in your Streamlit app
    """
    questions = []

    for i in range(num_questions):
        prompt = f"""### Passage:
{transcript.strip()}

### Task:
Generate question #{i+1} - a multiple-choice question based on the passage above.

### Question:"""

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

        with torch.no_grad():
            outputs = trainer.model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.8,  # Slightly higher for variety
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        parsed = parse_generated_question(generated_text)

        if parsed:
            questions.append(parsed)

    return questions

# Test
print("Generating 5 questions...\n")
test_questions = generate_questions(test_transcript, num_questions=5)

print("="*80)
print("GENERATED QUESTIONS:")
print("="*80)
for i, q in enumerate(test_questions, 1):
    print(f"\nQ{i}: {q['question']}")
    for j, choice in enumerate(q['choices']):
        marker = "✓" if j == q['correct'] else " "
        print(f"  {chr(65+j)}) {choice} {marker}")
print("="*80)

Generating 5 questions...

GENERATED QUESTIONS:


## 12. Save Integration Code

In [ ]:
# Save the integration functions for use in Streamlit
integration_code = '''
# INTEGRATION CODE FOR STREAMLIT APP
# Add this to your YouTube transcript app

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch
import json

# Load question generation model (run once at startup)
@st.cache_resource
def load_question_model():
    base_model_name = "mistralai/Mistral-7B-v0.1"
    model_path = "/path/to/question-generation-mistral"  # Update this

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model = PeftModel.from_pretrained(base_model, model_path)
    model.eval()

    return model, tokenizer

def generate_quiz_questions(model, tokenizer, transcript, num_questions=5):
    """Generate MCQ questions from transcript"""
    questions = []

    for i in range(num_questions):
        prompt = f"""### Passage:
{transcript}

### Task:
Generate question #{i+1} - a multiple-choice question.

### Question:"""

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.8,
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        parsed = parse_question(generated)

        if parsed:
            questions.append(parsed)

    return questions

# In your Streamlit app:
# 1. Load models
question_model, question_tokenizer = load_question_model()

# 2. After getting transcript
transcript = get_transcript(video_id)

# 3. Generate questions
questions = generate_quiz_questions(question_model, question_tokenizer, transcript)

# 4. Show quiz
for i, q in enumerate(questions):
    st.write(f"**Q{i+1}:** {q['question']}")
    answer = st.radio(f"q{i}", q['choices'], key=f"q{i}")
'''

with open('/content/drive/MyDrive/integration_code.py', 'w') as f:
    f.write(integration_code)

print("Integration code saved to Google Drive!")
print("File: /content/drive/MyDrive/integration_code.py")

Integration code saved to Google Drive!
File: /content/drive/MyDrive/integration_code.py


In [20]:
# Clear GPU memory first
import gc
del trainer, base_model
gc.collect()
torch.cuda.empty_cache()

print("Memory cleared. Reloading saved model...\n")

# Reload the SAVED model (not trainer.model)
from peft import PeftModel

model_path = "/content/drive/MyDrive/question-generation-mistral"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=bnb_config,
    device_map="auto",
)

print("Loading LoRA weights...")
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()

print("Model loaded!\n")

# NOW test
test_transcript = """
In this lecture, we'll discuss the concept of derivatives in calculus.
A derivative represents the rate of change of a function at a given point.
For example, if we have a function f(x) = x squared, the derivative is 2x.
This means that at any point x, the rate of change is twice the value of x.
Derivatives are fundamental in understanding how functions behave and are used
extensively in physics, engineering, and economics.
"""

prompt = f"""### Passage:
{test_transcript.strip()}

### Task:
Generate a multiple-choice question based on the passage above.

### Question:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Generating question...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.7,
        do_sample=True,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("="*80)
print("GENERATED:")
print("="*80)
print(generated)
print("="*80)

Memory cleared. Reloading saved model...

Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading LoRA weights...
Model loaded!

Generating question...
GENERATED:
### Passage:
In this lecture, we'll discuss the concept of derivatives in calculus.
A derivative represents the rate of change of a function at a given point.
For example, if we have a function f(x) = x squared, the derivative is 2x.
This means that at any point x, the rate of change is twice the value of x.
Derivatives are fundamental in understanding how functions behave and are used
extensively in physics, engineering, and economics.

### Task:
Generate a multiple-choice question based on the passage above.

### Question:
What is the function of derivatives?

### Choices:
A) To represent the rate of change of a function at a given point.
B) To show the rate of change of a function at a given point.
C) To show the rate of change of a function.
D) To represent the rate of change of a function.

### Correct Answer:
D


## Summary

**What was trained:**
- Model: Mistral 7B with LoRA
- Task: Generate MCQ questions from text passages
- Dataset: RACE (20K examples)
- Output format: Ready for Streamlit integration

**Next steps:**
1. Download model from Google Drive
2. Add integration code to your Streamlit app
3. Test with real video transcripts

**Model location:** `/content/drive/MyDrive/question-generation-mistral`